In [ ]:
!pip install fredapi

In [ ]:
# Connecting the Python Code with the google drive to access the datasets
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from fredapi import Fred
from google.colab import userdata

# =========================
# SETTINGS
# =========================

START_DATE = "1980-01-01"
FRED_API_KEY = userdata.get('FRED_API_KEY') # Retrieve API key from Colab Secrets
fred = Fred(api_key=FRED_API_KEY)

# =========================
# LOAD DATA
# =========================
m2 = fred.get_series("M2SL", observation_start=START_DATE).resample("ME").last()
cpi = fred.get_series("CPIAUCSL", observation_start=START_DATE).resample("ME").last()
gs10 = fred.get_series("GS10", observation_start=START_DATE).resample("ME").last()

lei = pd.read_excel("/content/drive/MyDrive/LEI.xlsx")
lei.columns = lei.columns.str.strip()
lei["Date"] = pd.to_datetime(lei["Date"])
lei = lei.set_index("Date").resample("ME").last()

sp_daily = yf.download("^GSPC", start=START_DATE, auto_adjust=True)["Close"].squeeze()
sp_daily.index = sp_daily.index.tz_localize(None)
sp = sp_daily.resample("ME").last()

# =========================
# BUILD DATAFRAME
# =========================
df = pd.DataFrame(index=sp.index)
df["sp"] = sp
df["monthly_ret"] = sp.pct_change()
df["fwd_6m_ret"] = sp.pct_change(6).shift(-6)

# =========================
# MACRO BLOCK
# =========================

# ISM vote
df["ism"] = lei["ISM Manufacturing New Orders"]
df["ism_3m_chg"] = df["ism"].diff(3)

df["ism_vote"] = 0
df.loc[((df["ism"] > 50) & (df["ism"].shift(1) <= 50)), "ism_vote"] = 1
df.loc[((df["ism"] < 50) & (df["ism_3m_chg"] >= 3)), "ism_vote"] = 1
df.loc[((df["ism"] > 50) & (df["ism_3m_chg"] > 0)), "ism_vote"] = 1

# real M2 impulse vote
df["raw_m2_yoy"] = m2.pct_change(12) * 100
df["cpi_yoy"] = cpi.pct_change(12) * 100
df["real_m2_yoy"] = df["raw_m2_yoy"] - df["cpi_yoy"]
df["real_m2_impulse"] = df["real_m2_yoy"].diff(6)

df["m2_vote"] = 0
df.loc[df["real_m2_impulse"] > 0, "m2_vote"] = 1
df.loc[df["real_m2_impulse"] < 0, "m2_vote"] = -1

# 10Y rates vote
df["rate_signal"] = (-gs10.diff(24)).shift(18)

df["rate_vote"] = 0
df.loc[df["rate_signal"] > 0, "rate_vote"] = 1
df.loc[df["rate_signal"] < 0, "rate_vote"] = -1

# macro score and regime
df["macro_score"] = df["ism_vote"] + df["m2_vote"] + df["rate_vote"]

df["macro_regime"] = "Neutral"
df.loc[df["macro_score"] >= 2, "macro_regime"] = "Bullish"
df.loc[df["macro_score"] <= -1, "macro_regime"] = "Bearish"

# =========================
# MOMENTUM BLOCK
# Bloomberg-style momentum:
# return from 180 days before to 14 days before calculation date
# =========================
mom_df = pd.DataFrame(index=sp.index)
mom_df["px_t_14"] = sp_daily.shift(14).resample("ME").last()
mom_df["px_t_180"] = sp_daily.shift(180).resample("ME").last()

df["momentum"] = mom_df["px_t_14"] / mom_df["px_t_180"] - 1
df["momentum_regime"] = "Neutral"
df.loc[df["momentum"] > 0, "momentum_regime"] = "Bullish"
df.loc[df["momentum"] < 0, "momentum_regime"] = "Bearish"

# =========================
# COMBINED SIGNAL
# =========================
df["signal"] = "HOLD"

# macro drives BUY
df.loc[df["macro_regime"] == "Bullish", "signal"] = "BUY"

# SELL only when macro bearish AND momentum bearish
df.loc[(df["macro_regime"] == "Bearish") & (df["momentum_regime"] == "Bearish"), "signal"] = "SELL"

# =========================
# SIGNAL SUMMARY
# =========================
signal_summary = (
    df.dropna(subset=["fwd_6m_ret"])
      .groupby("signal")["fwd_6m_ret"]
      .agg(["count", "mean", "median"])
      .round(4)
)

baseline = df["fwd_6m_ret"].mean()

print("SIGNAL SUMMARY")
print(signal_summary)
print("\nBaseline forward 6M return:", round(baseline, 4))

# =========================
# BACKTEST WITH SHORTING ON SELL
# BUY/HOLD = long, SELL = short
# =========================
df["position"] = 1.0
df.loc[df["signal"] == "SELL", "position"] = -1.0

df["strategy_ret"] = df["position"].shift(1) * df["monthly_ret"]
df["buyhold_ret"] = df["monthly_ret"]

bt = df.dropna(subset=["strategy_ret", "buyhold_ret"]).copy()

def annualized_return(ret):
    ret = ret.dropna()
    n_years = len(ret) / 12
    cum = (1 + ret).prod()
    return cum ** (1 / n_years) - 1

def annualized_vol(ret):
    return ret.dropna().std() * np.sqrt(12)

def sharpe(ret):
    vol = annualized_vol(ret)
    return annualized_return(ret) / vol if vol != 0 else np.nan

def max_dd(ret):
    cum = (1 + ret.dropna()).cumprod()
    return (cum / cum.cummax() - 1).min()

print("BACKTEST WITH SHORTING")
print("Strategy CAGR:", round(annualized_return(bt["strategy_ret"]), 4))
print("BuyHold CAGR:", round(annualized_return(bt["buyhold_ret"]), 4))
print("Strategy Sharpe:", round(sharpe(bt["strategy_ret"]), 4))
print("BuyHold Sharpe:", round(sharpe(bt["buyhold_ret"]), 4))
print("Strategy MaxDD:", round(max_dd(bt["strategy_ret"]), 4))
print("BuyHold MaxDD:", round(max_dd(bt["buyhold_ret"]), 4))
print("SELL frequency:", round((df["signal"] == "SELL").mean(), 4))

# =========================
# GROWTH OF $1 CHART
# =========================
bt["strategy_growth"] = (1 + bt["strategy_ret"]).cumprod()
bt["buyhold_growth"] = (1 + bt["buyhold_ret"]).cumprod()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=bt.index,
    y=bt["strategy_growth"],
    name="Macro + Momentum Model",
    mode="lines",
    line=dict(color="blue", width=2)
))

fig.add_trace(go.Scatter(
    x=bt.index,
    y=bt["buyhold_growth"],
    name="S&P500 Buy & Hold",
    mode="lines",
    line=dict(color="orange", width=2)
))

fig.update_layout(
    title="Growth of $1: Macro + Momentum vs Buy & Hold",
    xaxis_title="Date",
    yaxis_title="Growth of $1",
    plot_bgcolor="white",
    width=1000,
    height=500,
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.15)
)

fig.show()

In [ ]:
# =========================
# LATEST SIGNAL
# =========================
latest = df.dropna(subset=["macro_score"]).iloc[-1]

print("=" * 40)
print(f"S&P 500:         {latest['sp']:.2f}")
print("-" * 40)
print(f"ISM Vote:        {int(latest['ism_vote'])}")
print(f"M2 Vote:         {int(latest['m2_vote'])}")
print(f"Rate Vote:       {int(latest['rate_vote'])}")
print(f"Macro Score:     {int(latest['macro_score'])}  →  {latest['macro_regime']}")
print(f"Momentum:        {latest['momentum']:.4f}  →  {latest['momentum_regime']}")
print("-" * 40)
print(f"SIGNAL:          {latest['signal']}")
print("=" * 40)

In [ ]:
# =========================
# QUADRANT CHART
# =========================
import matplotlib.pyplot as plt
import matplotlib.patches as patches

latest = df.dropna(subset=["macro_score"]).iloc[-1]
macro = latest["macro_score"]
mom = latest["momentum"] * 100  # convert to %

fig, ax = plt.subplots(figsize=(8, 6))

# Zone backgrounds
# BUY: macro >= 2, full height
ax.axvspan(2, 3.5, color="green", alpha=0.08)
# SELL: macro <= -1 AND momentum < 0
ax.add_patch(patches.Rectangle((-2.5, -35), 1.5, 35, color="red", alpha=0.08))
# HOLD: everything else
ax.add_patch(patches.Rectangle((-2.5, 0), 1.5, 35, color="orange", alpha=0.05))
ax.axvspan(-1, 2, color="orange", alpha=0.05)

# Threshold lines
ax.axvline(x=2, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axvline(x=-1, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.plot([-2.5, -1], [0, 0], color="gray", linestyle="--", linewidth=0.8, alpha=0.5)

# Zone labels
ax.text(2.75, 28, "BUY", ha="center", fontsize=13, fontweight="bold", color="green", alpha=0.6)
ax.text(2.75, 24, "macro ≥ 2", ha="center", fontsize=9, color="green", alpha=0.5)
ax.text(-1.75, -18, "SELL", ha="center", fontsize=13, fontweight="bold", color="red", alpha=0.6)
ax.text(-1.75, -22, "macro ≤ -1\n& mom < 0", ha="center", fontsize=9, color="red", alpha=0.5)
ax.text(0.5, 28, "HOLD", ha="center", fontsize=13, fontweight="bold", color="orange", alpha=0.5)
ax.text(-1.75, 28, "HOLD", ha="center", fontsize=13, fontweight="bold", color="orange", alpha=0.5)

# Current position dot
ax.scatter(macro, mom, s=150, color="dodgerblue", edgecolors="navy", linewidths=2, zorder=5)
ax.annotate(
    f"  {latest.name.strftime('%b %Y')}\n  Macro: {int(macro)} | Mom: {mom:.1f}%\n  Signal: {latest['signal']}",
    xy=(macro, mom),
    xytext=(macro + 0.3, mom + 4),
    fontsize=9,
    color="navy",
    fontweight="bold",
)

# Axes
ax.set_xlim(-2.5, 3.5)
ax.set_ylim(-35, 35)
ax.set_xlabel("Macro score", fontsize=12)
ax.set_ylabel("Momentum (%)", fontsize=12)
ax.set_xticks(range(-2, 4))
ax.set_yticks(range(-30, 31, 10))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v)}%"))

ax.set_facecolor("white")
fig.patch.set_facecolor("white")
ax.grid(True, alpha=0.15)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()